# VadCLIP Temporal CNN: projection 256, 128 filters
Thử một cấu hình duy nhất: projection 512→256, CNN block đầu 256→128, block sau 128→128; kernel size 5, residual bật. Train from scratch với seed 234, đánh giá epoch 10. Không ghi đè các thí nghiệm cũ.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
OUTPUT_ROOT = PROJECT_ROOT / 'code' / 'results'
RUN_TAG = 'cnn_k5_p256_c128_s234'
VARIANT = 'k5_head_topk_p256'
SEED = 234
EPOCH = 10

if not torch.cuda.is_available():
    raise RuntimeError('Hãy chọn GPU runtime trên Colab.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Không tìm thấy UCFClipFeatures hoặc file nén trên Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
required = (SRC_DIR / 'model_cnn_vadclip.py', SRC_DIR / 'ucf_train_cnn_vadclip.py',
            SRC_DIR / 'ucf_evaluate_cnn_vadclip.py', SRC_DIR / 'ucf_cnn_common.py',
            LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
            LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy', FEATURE_ROOT)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print('GPU:', torch.cuda.get_device_name(0))
print('Feature root:', FEATURE_ROOT)
print('Run tag:', RUN_TAG)


## Kiểm tra kiến trúc
Kiểm tra shape, padding và gradient trước khi train.

In [ ]:
sys.path.insert(0, str(SRC_DIR))
from model_cnn_vadclip import TemporalCNNClassifier
cnn = TemporalCNNClassifier(projection_width=256, hidden_width=128, kernel_size=5, use_residual=True)
assert tuple(cnn.projection[0].weight.shape) == (256, 512)
assert tuple(cnn.blocks[0].conv1.weight.shape) == (128, 256, 5)
assert tuple(cnn.blocks[0].skip_projection.weight.shape) == (128, 256)
assert tuple(cnn.blocks[1].conv1.weight.shape) == (128, 128, 5)
scores = cnn(torch.randn(2, 20, 512), torch.tensor([13, 20]))
assert scores.shape == (2, 20, 1) and torch.isfinite(scores).all()
assert torch.equal(scores[0, 13:], torch.zeros_like(scores[0, 13:]))
scores[:, :13].sum().backward()
assert cnn.blocks[0].skip_projection.weight.grad is not None
print('Smoke test PASS: 512→256→128, kernel 5, residual on')
del cnn


## Train và đánh giá
Batch 64 Normal + 64 abnormal, LR 1e-5, milestone 4/8, 10 epoch. Chỉ đánh giá epoch 10; không chọn epoch theo AUC test.

In [ ]:
def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Tiến trình thất bại; traceback gốc đã in ở trên.')

checkpoint_dir = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_vadclip' / RUN_TAG / VARIANT
checkpoint = checkpoint_dir / f'epoch_{EPOCH:03d}.pth'
train_dir = OUTPUT_ROOT / 'train' / RUN_TAG / VARIANT
eval_dir = OUTPUT_ROOT / 'evaluate' / RUN_TAG / VARIANT
analyze_dir = OUTPUT_ROOT / 'analyze' / RUN_TAG / VARIANT
config_path = train_dir / 'config.json'
expected = {'projection_width': 256, 'cnn_channels': 128, 'kernel_size': 5,
            'score_mode': 'scoring_head', 'residual_mode': 'on', 'aggregation': 'topk',
            'lambda_alignment': 1.0, 'lambda_text': 1.0, 'seed': SEED,
            'batch_size': 64, 'lr': 1e-5, 'max_epoch': EPOCH,
            'scheduler_milestones': [4, 8], 'scheduler_rate': 0.1,
            'visual_length': 256, 'feature_root': str(FEATURE_ROOT),
            'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')}
if checkpoint.is_file():
    if not config_path.is_file():
        raise RuntimeError('Checkpoint tồn tại nhưng thiếu config. Hãy đổi RUN_TAG.')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    if any(config.get(key) != value for key, value in expected.items()):
        raise RuntimeError('Checkpoint cũ không đúng cấu hình. Hãy đổi RUN_TAG.')
    print('Đã có checkpoint đúng cấu hình, bỏ qua train:', checkpoint)
else:
    train_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py'),
                 '--feature-root', str(FEATURE_ROOT), '--train-list', expected['train_list'],
                 '--projection-width', '256', '--cnn-channels', '128',
                 '--kernel-size', '5', '--score-mode', 'scoring_head',
                 '--residual-mode', 'on', '--aggregation', 'topk',
                 '--lambda-alignment', '1.0', '--lambda-text', '1.0',
                 '--seed', str(SEED), '--batch-size', '64', '--lr', '1e-5',
                 '--max-epoch', str(EPOCH), '--scheduler-milestones', '4', '8',
                 '--scheduler-rate', '0.1', '--num-workers', '2',
                 '--visual-length', '256', '--epoch-checkpoint-dir', str(checkpoint_dir),
                 '--output-model-path', str(SRC_DIR / 'model' / 'cnn_vadclip' / RUN_TAG / f'{VARIANT}.pth'),
                 '--results-dir', str(train_dir)]
    run_command(train_cmd)
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
metrics_path = eval_dir / 'metrics.json'
if metrics_path.is_file():
    previous = json.loads(metrics_path.read_text(encoding='utf-8'))
    if previous.get('model_path') != str(checkpoint) or previous.get('projection_width') != 256:
        raise RuntimeError('Metrics cũ không đúng checkpoint/cấu hình. Hãy đổi RUN_TAG.')
    print('Đã có đánh giá đúng cấu hình, bỏ qua evaluate:', metrics_path)
else:
    eval_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py'),
                '--model-type', 'cnn', '--model-path', str(checkpoint),
                '--feature-root', str(FEATURE_ROOT),
                '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
                '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
                '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
                '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
                '--projection-width', '256', '--cnn-channels', '128',
                '--kernel-size', '5', '--score-mode', 'scoring_head',
                '--residual-mode', 'on', '--train-config-json', str(config_path),
                '--evaluate-dir', str(eval_dir), '--analyze-dir', str(analyze_dir)]
    run_command(eval_cmd)
print('Checkpoint:', checkpoint)
print('Train:', train_dir)
print('Evaluate:', eval_dir)
print('Analyze:', analyze_dir)


## So sánh
Đối chiếu P=256 với các kết quả P=128, C=128, k=5, residual bật đã lưu. Các lượt chạy trước có thể dao động theo seed và thứ tự dữ liệu.

In [ ]:
keys = ('classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP')
print('Model | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP')
for tag in ('cnn_vadclip_v1', 'cnn_vadclip_seed234_repeat'):
    old_path = OUTPUT_ROOT / 'evaluate' / tag / 'k5_head_topk' / 'metrics.json'
    if old_path.is_file():
        old = json.loads(old_path.read_text(encoding='utf-8'))
        print(tag + ' P128/C128 | ' + ' | '.join(f'{old[key]:.2f}' for key in keys))
    else:
        print('Không thấy kết quả tham chiếu:', old_path)
current = json.loads(metrics_path.read_text(encoding='utf-8'))
print('P256/C128 k5 residual on | ' + ' | '.join(f'{current[key]:.2f}' for key in keys))
summary_path = analyze_dir / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('Top-15 anomaly fraction (macro):', round(summary['selected_anomaly_fraction_abnormal_video_macro'], 3))
